# **Data Cleaning Notebook **

## Objectives

* Clean data in preparation for model training
  * Inspect missing values
  * Consider dropping features with high percentage of missing values
  * Impute missing values
  * Outlier treatment

## Inputs

outputs/datasets/collection/house_prices

## Outputs

Cleaned House Prices dataset - outputs/datasets/cleaned/house_prices_cleaned.csv
Cleaned train set - outputs/datasets/cleaned/train_set_cleaned.csv
Cleaned test set - outputs/datasets/cleaned/test_set_cleaned.csv


---

## Change working directory

In [ ]:
import os
current_dir = os.getcwd()
current_dir

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

In [ ]:
current_dir = os.getcwd()
current_dir

## Load Data


In [ ]:
import pandas as pd
df = pd.read_csv(f"outputs/datasets/collection/house_prices.csv")
df.head()

---

# Missing Values

We will inspect the dataset for missing values, and decide how to handle them.

From the visualisation, we can see that:
* Two variables, EnclosedPorch and WoodDeckSF have a very high percentage of missing values (90.7% and 89.4%) respectively.
* 3580 entries in total are missing, representing 10.2% of the dataset.

In [ ]:
# Display a table to visualise null data
null_data = []

for col in df.columns:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        null_data.append({
            'Variable': col,
            'Null Count': null_count,
            'Null %': round(null_count / len(df) * 100, 1),
            'Data Type': 'Numerical' if df[col].dtype in ['int64', 'float64'] else 'Categorical'
        })

null_df = pd.DataFrame(null_data).sort_values('Null %', ascending=False)

# Add totals row
total_null = null_df['Null Count'].sum()
total_pct = round(total_null / (len(df) * len(df.columns)) * 100, 1)

totals_row = pd.DataFrame([{
    'Variable': 'TOTAL',
    'Null Count': total_null,
    'Null %': total_pct,
    'Data Type': ''
}])

null_df = pd.concat([null_df, totals_row], ignore_index=True)
print(null_df.to_string(index=False))

### Dropping Missing Data

There is a very strong case for dropping EnclosedPorch and WoodDeckSF from the dataset entirely. 
Our reasoning for this is:

* The predictive value is going to be negligible if left as missing.
* Imputing such a high number of values would be unreliable and potentially misleading when training the model.
* Time and processing power would be saved by dropping. There's little point in carrying them through the pipeline. Although this dataset is small in the scale of enormous collections, it seems like best practice to do so. 

In [ ]:
# Drop variables with excessive missing values
cols_to_drop = ['EnclosedPorch', 'WoodDeckSF']
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns: {', '.join(cols_to_drop)}")
print(df.columns.tolist())

---

## Imputation

We will be imputing missing data for the following reasons:
* To avoid loss of statistical power, compared to dropping rows with null values.
* To reduce bias.
* Ensure compatability with algorithms.

We have chosen to impute median values for numerical values. The reasoning behind this is housing attributes data is likely to be skewed, with a small number of large properties affecting the mode. For this reason, we will be using median.

With categorical values, we will be imputing the mode value.



In [ ]:
from sklearn.impute import SimpleImputer

# Median imputation for numerical variables
median_imputer = SimpleImputer(strategy='median')
median_cols = ['LotFrontage', 'GarageYrBlt', 'BedroomAbvGr', '2ndFlrSF', 'MasVnrArea', 'OpenPorchSF']
df[median_cols] = median_imputer.fit_transform(df[median_cols])

# Mode imputation for categorical variables
mode_imputer = SimpleImputer(strategy='most_frequent')
mode_cols = ['BsmtExposure', 'BsmtFinType1', 'GarageFinish']
df[mode_cols] = mode_imputer.fit_transform(df[mode_cols])

# Confirm no null values remain
print(df.isnull().sum())

---

## Handling outliers

It is worth considering whether to drop or retain outliers when preparing data for model training. 

First we will explore and visualise outlying data using a histogram, and sceondly calculate the number of percentage and outliers.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# This code was taken and modified from Feature Engine Unit 6 of Code Institute's Full Stack program.
def plot_histogram_and_boxplot(df, col):
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(7, 7), 
                             gridspec_kw={"height_ratios": (.15, .85)})
    sns.boxplot(data=df, x=col, ax=axes[0])
    sns.histplot(data=df, x=col, kde=True, ax=axes[1])
    fig.suptitle(f"{col} Distribution - Boxplot and Histogram")
    plt.show()
    
    IQR = df[col].quantile(q=0.75) - df[col].quantile(q=0.25)
    print(
        f"This is the range where a datapoint is not an outlier: from "
        f"{(df[col].quantile(q=0.25) - 1.5*IQR).round(2)} to "
        f"{(df[col].quantile(q=0.75) + 1.5*IQR).round(2)}"
    )
    print("\n")

plot_histogram_and_boxplot(df, 'SalePrice')

In [ ]:
# Bound values taken from the output of the above function
lower_bound = 3937.5
upper_bound = 340037.5

outliers = df[(df['SalePrice'] < lower_bound) | (df['SalePrice'] > upper_bound)]

print(f"Number of outliers: {len(outliers)}")
print(f"Percentage of dataset: {round(len(outliers) / len(df) * 100, 1)}%")

We consider the following:
* The range where a datapoint is not an outlier: from 3937.5 to 340037.5
* Number of outliers: 61
* Percentage of dataset: 4.2%

4.2% is a small enough percentage to consider dropping, however we have decided to include outliers initially for the following reasons:
* Real life houses vary as reflected in the dataset, which we want to account for in our model.
* We are expecting to use a tree-based model, which handles outliers well.
* Dropping data would restrict our dataset.

We will return to this decision if we encounter issues with our model training.

# Push files to Repo

We will now save our cleaned data set 

In [ ]:
# Check dataframe before saving
df.info()

In [ ]:
# Create collections folder and save cleaned dataset
import os
try:
    os.makedirs(name='outputs/datasets/cleaned')
except Exception as e:
    print(e)

df.to_csv('outputs/datasets/cleaned/house_prices_cleaned.csv', index=False)

## Split Test and Train Sets

We will now split our dataset in preparation for Feature Engineering and Modelling.

In [ ]:
from sklearn.model_selection import train_test_split
TrainSet, TestSet, _, __ = train_test_split(
                                        df,
                                        df['SalePrice'],
                                        test_size=0.2,
                                        random_state=27)

print(f"TrainSet shape: {TrainSet.shape} \nTestSet shape: {TestSet.shape}")

### Save train set to outputs folder

In [ ]:
TrainSet.to_csv("outputs/datasets/cleaned/train_set_cleaned.csv", index=False)

### Save test set to outputs folder

In [ ]:
TestSet.to_csv("outputs/datasets/cleaned/test_set_cleaned.csv", index=False)